In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Titanic training dataset from Kaggle path
df = pd.read_csv("/kaggle/input/titanic/train.csv")
df.head()


In [ ]:
#Data Pre-Processing
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# Handling missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])          # female = 0, male = 1
df['Embarked'] = le.fit_transform(df['Embarked'])  # C/Q/S → 0/1/2

X = df.drop('Survived', axis=1)
y = df['Survived']

print("Data preprocessing completed.")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("\nPreview of processed data:")
display(df.head())

In [ ]:
#Fine-Tune and Plot Decision Tree
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import numpy as np

# Defining parameters for hyperparameter tuning
param_grid = {
    "max_depth": [3, 4, 5, 6, 8, None],
    "min_samples_split": [2, 4, 6, 8],
    "criterion": ["gini", "entropy"]
}

# Initialize Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Perform Grid Search with 5-fold CV
grid_dt = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X, y)

# Best model
best_dt = grid_dt.best_estimator_
print("Best Decision Tree Parameters:", grid_dt.best_params_)
print("Best Cross-Validation Accuracy:", round(grid_dt.best_score_, 4))

# Plot the fine-tuned tree
plt.figure(figsize=(18, 10))
plot_tree(
    best_dt,
    feature_names=X.columns,
    class_names=['Died', 'Survived'],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Fine-Tuned Decision Tree", fontsize=14)
plt.show()


In [ ]:
#5-Fold CV and Model Comparison
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Decision Tree 5-Fold CV
scores_dt = cross_val_score(best_dt, X, y, cv=5, scoring='accuracy')
print("Decision Tree 5-fold accuracies:", scores_dt)
print("Average Decision Tree Accuracy:", np.mean(scores_dt).round(4))

#Random Forest Fine-Tuning
param_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [4, 6, 8, None],
    "min_samples_split": [2, 4, 6],
    "criterion": ["gini", "entropy"]
}

rf = RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(rf, param_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X, y)

best_rf = grid_rf.best_estimator_
print("\n Best Random Forest Parameters:", grid_rf.best_params_)

#Random Forest 5-Fold CV
scores_rf = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy')
print("Random Forest 5-fold accuracies:", scores_rf)
print("Average Random Forest Accuracy:", np.mean(scores_rf).round(4))
print("\n Algorithm Comparison ")
print("Decision Tree Avg Accuracy:", np.mean(scores_dt).round(4))
print("Random Forest Avg Accuracy:", np.mean(scores_rf).round(4))

if np.mean(scores_rf) > np.mean(scores_dt):
    print("\n Random Forest performed better — more robust and less overfitting.")
else:
    print("\n Decision Tree performed better — simpler model suited for this data.")
